In [ ]:
# Script for styling the Jupyter Notebook using an external CSS file
from IPython.display import HTML

with open("style.css", "r") as f:
    css = f.read()

HTML(f"<style>{css}</style>")

# SAGE model training

This notebook generates a training report for the GraphConv model, covering three experiments:

1. Hyperparameter optimization — selecting the best-performing model configuration.
2. Sample size optimization — quantifying the minimum number of samples needed to improve accuracy.
3. Full-sample model — training with all available data.

Results directory: `/home/mriveraceron/glv-research/tuning_results/SAGE_hpo`

Data directory: `/home/mriveraceron/glv-research/data_null/`

# Hyperparameter optimization

In [ ]:
# Display results table
import pandas as pd
from IPython.display import display, HTML

# Experiment directory
dir = '/home/mriveraceron/glv-research/updated_results/SAGE_hpo'
df = pd.read_csv(f'{dir}/tuning_results.csv')
display(df.sort_values(by="pearson_corr", ascending=False)
          .style.set_table_styles([{'selector': '', 'props': [('margin', '0 auto')]}]))

In [ ]:
# Define the plot style function
import matplotlib.pyplot as plt

# Add the font directory directly
import matplotlib.font_manager as fm
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman.ttf')
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman Bold.ttf')
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman Italic.ttf')
fm.fontManager.addfont('/home/mriveraceron/fonts_git/Times New Roman Bold Italic.ttf')

plt.rcParams.update({
    'font.family':  'serif',
    'font.serif':   'Times New Roman',
    'font.size':    12,                  # sets the global base size
    'axes.facecolor':   'white',        # ← add this
    'figure.facecolor': 'white',        # ← and this (covers fig too)
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'grid.linewidth':   0.8,
    'grid.alpha':       0.7,
})

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Legend handles — built once, shared across all columns
metrics = ['ppv_idx', 'pearson_corr', 'spearman_corr']
titles  = ['Valor Predictivo Positivo (VPP)', 'Correlación de Pearson', 'Correlación de Spearman']

# --- Pre-process ---
df[metrics] = df[metrics].fillna(0)
combos      = df[['channels', 'layers']].drop_duplicates().reset_index(drop=True)
enum_combos = list(combos.iterrows())  # single enumeration reused below

# --- Mappings ---
lr_labels  = np.sort(df['learning_rate'].unique())[::-1]
lr_map     = {lr: i for i, lr in enumerate(lr_labels)}
palette    = plt.cm.tab10.colors
offsets    = np.linspace(-0.2, 0.2, len(combos))
color_map  = {(int(r['channels']), int(r['layers'])): palette[k] for k, (_, r) in enum_combos}
offset_map = {(int(r['channels']), int(r['layers'])): offsets[k] for k, (_, r) in enum_combos}
handles    = [
    mpatches.Patch(color=palette[k], label=f"n={int(r['channels'])} | c={int(r['layers'])}")
    for k, (_, r) in enum_combos
]

# --- Precompute keys once ---
df['_key'] = list(zip(df['channels'].astype(int), df['layers'].astype(int)))
df['_x']   = df.apply(lambda r: lr_map[r['learning_rate']] + offset_map[r['_key']], axis=1)
df['_color'] = df['_key'].map(color_map)

# --- Plot ---
plt.close('all')
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
for ax, metric, title, letter in zip(axes, metrics, titles, ['A', 'B', 'C']):
    for _, row in df.iterrows():
        color, x, val = row['_color'], row['_x'], row[metric]
        ax.plot([x, x], [0, val], color=color, lw=2, alpha=0.6)
        ax.scatter(x, val, color=color, s=100, zorder=5, edgecolors='white', linewidths=0.8)
        ax.text(x, val + 0.01, f'{val:.2f}', ha='center', va='bottom', fontsize=7, color=color)
    ax.set_xticks(range(len(lr_labels)))
    ax.set_xticklabels([f'{lr:.0e}' for lr in lr_labels], fontsize=12)
    ax.set_ylabel('Valor de la métrica', fontsize=12, labelpad=10)
    ax.set_xlabel('Tasa de aprendizaje', fontsize=12, labelpad=10)
    ax.set_ylim(-0.5, 0.5)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    ax.spines[['top', 'right']].set_visible(False)
    ax.text(-0.05, 1.05, letter, transform=ax.transAxes, fontsize=12, fontweight='bold', va='bottom')
    ax.legend(handles=handles, title='Neuronas | Capas', loc='upper left', frameon=True, framealpha=0.4, fontsize=10, title_fontsize=10)
  

plt.suptitle('Figura 11: Desempeño del modelo SAGE a diferente configuración de hiperparámetros', fontweight='bold')
plt.tight_layout()  # leaves 5% headroom at the top for the suptitle
plt.subplots_adjust(bottom=0.15)   # make room for the legend
plt.savefig('/home/mriveraceron/glv-research/thesis_plots/SAGE_hpo_panel.png')
plt.show()

# Sample size optimization

Results directory: `/home/mriveraceron/glv-research/tuning_results/SAGE_ss_V2`

Data directory: `/home/mriveraceron/glv-research/data_null`

In [ ]:
# Display results table
import pandas as pd
from IPython.display import display, HTML


# Experiment directory
dir = '/home/mriveraceron/glv-research/updated_results/SAGE_ss'
df = pd.read_csv(f'{dir}/tuning_results.csv')
display(df.sort_values(by="pearson_corr", ascending=False).style.set_table_styles([{'selector': '', 'props': [('margin', '0 auto')]}]))

In [ ]:
# Elapsed time plot
import matplotlib.pyplot as plt

# --- Config ---
save_path      = '/home/mriveraceron/glv-research/thesis_plots/SAGE_ss_panel.png'
COLOR_LINE     = '#2166ac'
COLORS_BAR     = ['#E69F00', '#56B4E9', '#009E73']
METRICS        = {
    'ppv_idx':       'VPP',
    'pearson_corr':  'Corr. Pearson',
    'spearman_corr': 'Corr. Spearman',
}
WIDTH  = 0.25
OFFSETS = [-WIDTH, 0, WIDTH]

# --- Data ---
x_num = df['train_size']
x_cat = df['train_size'].astype(str)
x_pos = range(len(x_cat))

# --- Figure ---
plt.close('all')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Figura 12: Desempeño de la mejor variante de SAGE por tamaño de muestra', fontweight='bold', fontsize=12)

# --- Panel labels ---
for ax, letter in zip([ax1, ax2], ['A', 'B']):
    ax.text(-0.05, 1.05, letter, transform=ax.transAxes, fontsize=12, fontweight='bold', va='bottom', ha='right')
    ax.spines[['top', 'right']].set_visible(False)

# --- A: Elapsed time ---
y = df['elapsed(s)']
ax1.plot(x_num, y, color=COLOR_LINE, linewidth=2.2, zorder=3)
ax1.fill_between(x_num, y, alpha=0.08, color=COLOR_LINE)
ax1.scatter(x_num, y, color=COLOR_LINE, s=50, zorder=4)
ax1.set_ylabel('Tiempo transcurrido (segundos)', labelpad=8, fontsize=12)
ax1.set_xlabel('Tamaño de muestra para entrenamiento', labelpad=8, fontsize=12)
for xi, yi in zip(x_num, y):
    ax1.annotate(f'{yi:.1f}s', xy=(xi, yi), xytext=(0, 8), textcoords='offset points',
                 ha='center', va='bottom', fontsize=8, color=COLOR_LINE)

# --- B: Metrics ---
for (metric, label), color, offset in zip(METRICS.items(), COLORS_BAR, OFFSETS):
    bars = ax2.bar(
        [p + offset for p in x_pos], df[metric],
        width=WIDTH, label=label, color=color,
        alpha=0.88, edgecolor='white', linewidth=0.6, zorder=3
    )
    ax2.bar_label(bars, fmt='%.2f', fontsize=7.5, padding=3, rotation=90, color='#333333')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(x_cat, rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('Valor de la métrica', labelpad=8, fontsize=12)
ax2.set_xlabel('Tamaño de muestra para entrenamiento', labelpad=8, fontsize=12)
ax2.set_ylim(0, 0.5)
ax2.legend(title='Métrica', loc='upper left', frameon=True, framealpha=0.5, edgecolor='#cccccc', fontsize=9, title_fontsize=10)

plt.tight_layout()
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()